# Search-11d — Descente sous budget : la loi derrière les métaheuristiques

**Série** : Search / Part1-Foundations — variante -d de [Search-11](Search-11-Metaheuristics.ipynb) (métaheuristiques MEALPy) et [Search-11c](Search-11c-Empirical-Algorithm-Selection.ipynb) (sélection empirique)
**Opération étudiée** : **op 11 « Descendre sous budget »**, après `mimo_lean/Descent.lean` (Lean-formel)
**Source primaire (lue firsthand)** : `MyIA.AI.Notebooks/SymbolicAI/Lean/mimo_lean/Descent.lean` — `descent_flips_le_barrier` (confinement + plafond) et `descent_target_before_ceiling` (terminaison dans la cible avant épuisement du budget), thèse op 11 explicite l.148

**Ce que ce notebook fait que Search-11 ne fait pas** : Search-11 *utilise* des métaheuristiques qui améliorent iterativement sous un budget d'évaluations ; il n'énonce ni ne mesure jamais la **loi** qui garantit qu'elles terminent. Ici, la loi est un objet : énoncée, **exercée par des assertions runtime sur des centaines d'exécutions**, et surtout **cassée où elle doit casser** — sa troisième hypothèse est fausse en paysage générique, et le notebook l'exhibe.

**Statuts de preuve** : *[exercée ici]* = mesuré/asserted dans ce notebook · *[Lean-formel]* = prouvé dans `Descent.lean` · *[rapporté]* = affirmé sans re-vérification.

Exécution : std uniquement, CPU, ~20 min de lecture, ~5 s d'exécution.

**Navigation** : [Search-11c](Search-11c-Empirical-Algorithm-Selection.ipynb) | [Version C#](Search-11d-Descente-Sous-Budget-CSharp.ipynb) | [Search-12](Search-12-BranchAndBound-Borne-LP-Python.ipynb)

## 1. La loi, et ce que l'attestation Lean prouve exactement

`Descent.lean` formalise une descente comme une suite d'états où seuls les flips **acceptés** comptent, sous trois hypothèses :

| Hypothèse | Énoncé | Rôle |
|---|---|---|
| `hstrict` | chaque mouvement accepté décroît **strictement** le coût | interdit les cycles |
| `hbarrier` | le coût reste **confiné** sous une borne `B` sur tout le run | majore le nombre de pas |
| `hnostall` | hors de la cible, un mouvement accepté existe **toujours** | bloque uniquement sur la cible |

Sous ces trois hypothèses, `descent_target_before_ceiling` démontre : le dernier état est **dans la cible**, et le run a effectué **strictement moins de flips acceptés que le plafond** `M_N` dès que `B < M_N`. C'est la garantie de complexité : borne de flips pour un run terminal ; ce plafond n'est pas un budget d'évaluations.

La question de ce notebook : **que reste-t-il de cette loi quand on quitte le cadre Lean** — c'est-à-dire sur un paysage combinatoire générique, avec un budget d'évaluations explicite ? Réponse en trois temps : la loi de terminaison **tient** (§2, exercée), la troisième hypothèse **tombe** (§3, exhibée), et la qualité est mesurée selon le budget (§4).

In [2]:
println!("Rust, std only");

Rust, std only


## 2. L'objet : une descente sous budget, instrumentée

Le substrat : **sac à dos 0/1** (instances synthétiques seedées). Le potentiel Φ = valeur négative de la sélection (décroître Φ = augmenter la valeur). Le voisinage : flips de bit uniques. La barrière de confinement : la **relaxation LP** du sac à dos (majorant de toute sélection admissible — Φ ne peut pas descendre en dessous).

Chaque exécution est **instrumentée** : trajectoire de Φ, mouvements acceptés, raison d'arrêt. Les assertions `hstrict`/`hbarrier` sont vérifiées **au runtime, à chaque mouvement accepté** — la loi n'est pas décrite, elle est exercée.

Pour relier le modèle au coût naturel de Lean, on peut prendre `floor(lp) - valeur(x)` : les valeurs sont entières et admissibles. Le nombre de flips acceptés est alors borné par ce coût initial, tandis que les évaluations comptent aussi les voisins rejetés. Un arrêt sur budget ne certifie pas la terminalité exigée par Lean.

In [3]:
// Générateur MT19937 compatible CPython (init_by_array sur les mots 32 bits du seed)
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) } // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 { // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 { // i continue là où la boucle précédente l'a laissé
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 { // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 { // 1 <= k <= 64 ; mots petit-boutistes, le dernier tronque
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 { // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

// Instance 0/1 synthetique : n objets, capacite = 0.5 * somme poids
fn instance_knapsack(n: usize, seed: i64) -> (Vec<i64>, Vec<i64>, i64) {
    let mut rng = Mt19937::new(seed as u32);
    let poids: Vec<i64> = (0..n).map(|_| rng.randint(10, 100)).collect();
    let valeurs: Vec<i64> = (0..n).map(|i| poids[i] + rng.randint(-5, 20)).collect(); // correlation valeur/poids
    let capacite = (0.5 * poids.iter().sum::<i64>() as f64) as i64;
    (poids, valeurs, capacite)
}

// Relaxation fractionnaire (greedy densite) : majorant de l'optimum 0/1
fn borne_lp(poids: &[i64], valeurs: &[i64], capacite: i64) -> f64 {
    let n = poids.len();
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&a, &b| (valeurs[b] as f64 / poids[b] as f64).total_cmp(&(valeurs[a] as f64 / poids[a] as f64)));
    let mut reste = capacite;
    let mut total = 0.0f64;
    for &i in &ordre {
        let pris = std::cmp::min(poids[i], reste);
        total += (valeurs[i] * pris) as f64 / poids[i] as f64;
        reste -= pris;
        if reste == 0 { break; }
    }
    total
}

// Optimum 0/1 exact par programmation dynamique (temoin de verite)
fn dp_exact(poids: &[i64], valeurs: &[i64], capacite: i64) -> i64 {
    let mut best = vec![0i64; (capacite + 1) as usize];
    for (p, v) in poids.iter().zip(valeurs.iter()) {
        for c in (*p as usize..=capacite as usize).rev() {
            if best[c - *p as usize] + v > best[c] {
                best[c] = best[c - *p as usize] + v;
            }
        }
    }
    best[capacite as usize]
}

fn admissible(x: &[i64], poids: &[i64], capacite: i64) -> bool {
    poids.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(p, &_)| *p).sum::<i64>() <= capacite
}

fn valeur(x: &[i64], valeurs: &[i64]) -> i64 {
    valeurs.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(&v, _)| v).sum()
}

struct Descente {
    x: Vec<i64>,
    valeur: i64,
    trajectoire: Vec<i64>,
    pas: i64,
    evaluations: i64,
    raison: &'static str,
    hstrict_viol: i64,
    hbarrier_viol: i64,
    lp: f64,
}

// Hill-climbing a stricte amelioration, plafonne a `budget` evaluations
fn descente_sous_budget(x0: &[i64], poids: &[i64], valeurs: &[i64], capacite: i64, budget: i64, collect: bool) -> Descente {
    let lp = borne_lp(poids, valeurs, capacite);
    let mut x = x0.to_vec();
    let mut phi = -valeur(&x, valeurs); // potentiel : decroitre = s'ameliorer
    let mut trajectoire: Vec<i64> = if collect { vec![phi] } else { vec![] };
    let mut hstrict_viol = 0i64;
    let mut hbarrier_viol = 0i64;
    let mut evaluations = 0i64;
    let mut pas = 0i64;
    let mut blocage = false;

    while evaluations < budget {
        let mut best_delta = 0i64;
        let mut best_i: Option<usize> = None;
        let mut explores = 0i64;
        for i in 0..x.len() {
            if evaluations >= budget { break; }
            explores += 1;
            let mut y = x.clone();
            y[i] = 1 - y[i];
            if !admissible(&y, poids, capacite) {
                evaluations += 1;
                continue;
            }
            evaluations += 1;
            let d = valeur(&y, valeurs) - valeur(&x, valeurs);
            if d > best_delta {
                best_delta = d;
                best_i = Some(i);
            }
        }
        match best_i {
            None => {
                // Un balayage incomplet ne prouve pas l'absence d'ameliorant.
                if explores == x.len() as i64 { blocage = true; }
                break;
            }
            Some(bi) => {
                let mut y = x.clone();
                y[bi] = 1 - y[bi];
                let phi_new = -valeur(&y, valeurs);
                if phi_new >= phi { hstrict_viol += 1; } // hstrict : mouvement accepte doit decroitre strictement
                if (-phi_new as f64) > lp + 1e-9 { hbarrier_viol += 1; } // hbarrier : confinement sous le majorant LP
                x = y;
                phi = phi_new;
                pas += 1;
                if collect { trajectoire.push(phi); }
            }
        }
    }
    // budget epuise avant balayage complet du voisinage
    let raison = if blocage { "blocage" } else { "budget" }; // le dernier balayage incomplet peut cacher un ameliorant : le noter honnetement

    Descente {
        x,
        valeur: -phi,
        trajectoire,
        pas,
        evaluations,
        raison,
        hstrict_viol,
        hbarrier_viol,
        lp,
    }
}

println!("Fonctions sac à dos initialisées : LP, DP exacte, admissibilité, valeur et descente instrumentée.");

Fonctions sac à dos initialisées : LP, DP exacte, admissibilité, valeur et descente instrumentée.


### Lecture de l'instrumentation

- `raison` distingue **budget atteint** (le plafond d'évaluations a interrompu la recherche alors qu'un améliorant pouvait exister) de **blocage** (un balayage **complet** du voisinage n'a trouvé aucun mouvement améliorant) — la dissociation que `Descent.lean` érige en objet.
- `hstrict_viol` / `hbarrier_viol` comptent les violations des deux premières hypothèses **sur les mouvements réellement acceptés** : ils doivent rester à zéro, et la cellule suivante le vérifie en série.
- `lp` est le majorant LP : la barrière de confinement. Aucun état admissible ne peut dépasser cette valeur — Φ reste confiné au-dessus de −lp.

In [4]:
// LA LOI, EXERCICEE : hstrict + hbarrier + plafond, sur 60 executions

struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn instance_knapsack(n: usize, seed: i64) -> (Vec<i64>, Vec<i64>, i64) {
    let mut rng = Mt19937::new(seed as u32);
    let poids: Vec<i64> = (0..n).map(|_| rng.randint(10, 100)).collect();
    let valeurs: Vec<i64> = (0..n).map(|i| poids[i] + rng.randint(-5, 20)).collect();
    let capacite = (0.5 * poids.iter().sum::<i64>() as f64) as i64;
    (poids, valeurs, capacite)
}

fn borne_lp(poids: &[i64], valeurs: &[i64], capacite: i64) -> f64 {
    let n = poids.len();
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&a, &b| (valeurs[b] as f64 / poids[b] as f64).total_cmp(&(valeurs[a] as f64 / poids[a] as f64)));
    let mut reste = capacite;
    let mut total = 0.0f64;
    for &i in &ordre {
        let pris = std::cmp::min(poids[i], reste);
        total += (valeurs[i] * pris) as f64 / poids[i] as f64;
        reste -= pris;
        if reste == 0 { break; }
    }
    total
}

fn dp_exact(poids: &[i64], valeurs: &[i64], capacite: i64) -> i64 {
    let mut best = vec![0i64; (capacite + 1) as usize];
    for (p, v) in poids.iter().zip(valeurs.iter()) {
        for c in (*p as usize..=capacite as usize).rev() {
            if best[c - *p as usize] + v > best[c] {
                best[c] = best[c - *p as usize] + v;
            }
        }
    }
    best[capacite as usize]
}

fn admissible(x: &[i64], poids: &[i64], capacite: i64) -> bool {
    poids.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(p, &_)| *p).sum::<i64>() <= capacite
}

fn valeur(x: &[i64], valeurs: &[i64]) -> i64 {
    valeurs.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(&v, _)| v).sum()
}

struct Descente {
    x: Vec<i64>,
    valeur: i64,
    trajectoire: Vec<i64>,
    pas: i64,
    evaluations: i64,
    raison: &'static str,
    hstrict_viol: i64,
    hbarrier_viol: i64,
    lp: f64,
}

fn descente_sous_budget(x0: &[i64], poids: &[i64], valeurs: &[i64], capacite: i64, budget: i64, collect: bool) -> Descente {
    let lp = borne_lp(poids, valeurs, capacite);
    let mut x = x0.to_vec();
    let mut phi = -valeur(&x, valeurs);
    let mut trajectoire: Vec<i64> = if collect { vec![phi] } else { vec![] };
    let mut hstrict_viol = 0i64;
    let mut hbarrier_viol = 0i64;
    let mut evaluations = 0i64;
    let mut pas = 0i64;
    let mut blocage = false;

    while evaluations < budget {
        let mut best_delta = 0i64;
        let mut best_i: Option<usize> = None;
        let mut explores = 0i64;
        for i in 0..x.len() {
            if evaluations >= budget { break; }
            explores += 1;
            let mut y = x.clone();
            y[i] = 1 - y[i];
            if !admissible(&y, poids, capacite) {
                evaluations += 1;
                continue;
            }
            evaluations += 1;
            let d = valeur(&y, valeurs) - valeur(&x, valeurs);
            if d > best_delta {
                best_delta = d;
                best_i = Some(i);
            }
        }
        match best_i {
            None => {
                if explores == x.len() as i64 { blocage = true; }
                break;
            }
            Some(bi) => {
                let mut y = x.clone();
                y[bi] = 1 - y[bi];
                let phi_new = -valeur(&y, valeurs);
                if phi_new >= phi { hstrict_viol += 1; }
                if (-phi_new as f64) > lp + 1e-9 { hbarrier_viol += 1; }
                x = y;
                phi = phi_new;
                pas += 1;
                if collect { trajectoire.push(phi); }
            }
        }
    }
    let raison = if blocage { "blocage" } else { "budget" };

    Descente {
        x,
        valeur: -phi,
        trajectoire,
        pas,
        evaluations,
        raison,
        hstrict_viol,
        hbarrier_viol,
        lp,
    }
}

let mut runs: Vec<(usize, i64, Descente)> = Vec::new();
for n in [16, 24, 32] {
    for seed in 0..20 {
        let (poids, valeurs, capacite) = instance_knapsack(n, seed);
        let x0 = vec![0i64; n];
        let budget = (8 * n * n) as i64; // generosite : budget >> longueur typique d'un run
        let r = descente_sous_budget(&x0, &poids, &valeurs, capacite, budget, true);
        runs.push((n, seed, r));
    }
}

let v_strict: i64 = runs.iter().map(|(_, _, r)| r.hstrict_viol).sum();
let v_barrier: i64 = runs.iter().map(|(_, _, r)| r.hbarrier_viol).sum();
let depassements = runs.iter().filter(|(n, _, r)| r.evaluations > (8 * *n * *n) as i64).count();

println!("executions              : {}", runs.len());
println!("violations hstrict      : {}   (mouvements acceptes sans decroissance stricte)", v_strict);
println!("violations hbarrier     : {}   (etat au-dela du majorant LP)", v_barrier);
println!("depassements de plafond : {}   (evaluations > budget)", depassements);
assert!(v_strict == 0 && v_barrier == 0 && depassements == 0);
for (_, _, r) in &runs {
    assert!(r.trajectoire.len() == r.pas as usize + 1);
    assert!(r.trajectoire.windows(2).all(|w| w[1] < w[0]));
    assert!(r.pas <= (r.lp as i64) + r.trajectoire[0]);
    assert!(r.trajectoire.iter().all(|&phi| (-phi as f64) <= r.lp + 1e-9));
}
println!("=> *[exercee ici]* la loi de terminaison tient : strictement decroissant, confine, plafonne.");

// Un balayage partiel n'est pas un certificat de blocage.
let a = descente_sous_budget(&[0, 0], &[2, 1], &[1, 2], 1, 1, true);
let b = descente_sous_budget(&[0, 0], &[2, 1], &[1, 2], 1, 0, true);
let c = descente_sous_budget(&[0, 1], &[2, 1], &[1, 2], 1, 2, true);
assert!((a.raison, b.raison, c.raison) == ("budget", "budget", "blocage"));
println!("fixtures partielle, zero et balayage complet : OK");

// Depart non nul : borne de flips calculee depuis la valeur initiale.
let t = descente_sous_budget(&[1, 0], &[1, 1], &[1, 4], 2, 10, true);
let t_sans = descente_sous_budget(&[1, 0], &[1, 1], &[1, 4], 2, 10, false);
assert!(t.trajectoire == vec![-1, -5] && t.pas == 1);
assert!(t.pas <= (t.lp as i64) - 1 && t_sans.trajectoire.is_empty());
println!("trajectoire, depart non nul et collect=False : OK");

executions              : 60


violations hstrict      : 0   (mouvements acceptes sans decroissance stricte)


violations hbarrier     : 0   (etat au-dela du majorant LP)


depassements de plafond : 0   (evaluations > budget)


=> *[exercee ici]* la loi de terminaison tient : strictement decroissant, confine, plafonne.


fixtures partielle, zero et balayage complet : OK


trajectoire, depart non nul et collect=False : OK


### Interprétation

Les compteurs ci-dessus contrôlent les mouvements effectivement acceptés et le budget d'évaluations sur ces 60 runs. La borne naturelle `floor(lp) - valeur(x0)` majore les flips acceptés ; elle ne majore pas directement le nombre d'évaluations. La terminaison sous budget vient ici du compteur explicite, et un arrêt sur budget ne prouve pas la terminalité de Lean.

La décroissance stricte du potentiel et son confinement restent des contrôles distincts : le premier exclut les cycles de mouvements acceptés, le second borne leur nombre. Un balayage peut cependant essayer plusieurs voisins avant d'accepter un flip ; le budget d'évaluations se consomme aussi sur les essais rejetés. Le transport de la borne Lean porte donc sur les flips et sur ses hypothèses, tandis que l'arrêt effectif de cette implémentation est assuré par son budget explicite.

### Exercice 1 — redémarrages stratégiques

La section suivante étudiera les blocages locaux. Répartir un budget global entre `k_restarts` descentes, en demandant chaque point initial à `x0_gen` et en appelant le callback `descendre(x, budget_partiel)` pour chacun. Retourner le meilleur objet obtenu de ce callback, sans le modifier. Le harnais compte les appels et budgets réellement alloués, y compris budget nul et capacité nulle. Une fois le stub complété, il mesure les 60 instances de la section 3, avec et sans quatre redémarrages, au même budget global de 2304 évaluations. Aucun gain n'est supposé.

**Méthode.** Garder les mêmes poids, valeurs et capacité pour les comparaisons : seuls les points de départ et l'allocation du budget changent. Conserver chaque objet retourné par la descente et choisir celui dont la valeur est maximale ; les évaluations déjà payées lors d'un redémarrage restent imputées au budget global. Le contrôle observe les callbacks pour distinguer une véritable série de descentes d'une valeur construite sans exécution.

**Lecture attendue.** Comparer les valeurs et les raisons d'arrêt, puis expliquer si le partage du budget permet de quitter certains bassins locaux. Plusieurs essais plus courts peuvent explorer davantage de bassins, mais aussi manquer de temps pour progresser dans chacun : une amélioration sur toutes les graines n'est pas une conséquence du redémarrage. Le cas de budget nul doit produire un résultat cohérent sans dépenser d'évaluation.

In [5]:
// Stub C.1 — Exercice 1 : redémarrages stratégiques

struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn instance_knapsack(n: usize, seed: i64) -> (Vec<i64>, Vec<i64>, i64) {
    let mut rng = Mt19937::new(seed as u32);
    let poids: Vec<i64> = (0..n).map(|_| rng.randint(10, 100)).collect();
    let valeurs: Vec<i64> = (0..n).map(|i| poids[i] + rng.randint(-5, 20)).collect();
    let capacite = (0.5 * poids.iter().sum::<i64>() as f64) as i64;
    (poids, valeurs, capacite)
}

fn borne_lp(poids: &[i64], valeurs: &[i64], capacite: i64) -> f64 {
    let n = poids.len();
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&a, &b| (valeurs[b] as f64 / poids[b] as f64).total_cmp(&(valeurs[a] as f64 / poids[a] as f64)));
    let mut reste = capacite;
    let mut total = 0.0f64;
    for &i in &ordre {
        let pris = std::cmp::min(poids[i], reste);
        total += (valeurs[i] * pris) as f64 / poids[i] as f64;
        reste -= pris;
        if reste == 0 { break; }
    }
    total
}

fn dp_exact(poids: &[i64], valeurs: &[i64], capacite: i64) -> i64 {
    let mut best = vec![0i64; (capacite + 1) as usize];
    for (p, v) in poids.iter().zip(valeurs.iter()) {
        for c in (*p as usize..=capacite as usize).rev() {
            if best[c - *p as usize] + v > best[c] {
                best[c] = best[c - *p as usize] + v;
            }
        }
    }
    best[capacite as usize]
}

fn admissible(x: &[i64], poids: &[i64], capacite: i64) -> bool {
    poids.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(p, &_)| *p).sum::<i64>() <= capacite
}

fn valeur(x: &[i64], valeurs: &[i64]) -> i64 {
    valeurs.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(&v, _)| v).sum()
}

fn median(v: &mut Vec<f64>) -> f64 {
    v.sort_by(|a, b| a.total_cmp(b));
    let n = v.len();
    if n % 2 == 1 { v[n / 2] } else { (v[n / 2 - 1] + v[n / 2]) / 2.0 }
}

#[derive(Clone)]
struct Descente {
    x: Vec<i64>,
    valeur: i64,
    trajectoire: Vec<i64>,
    pas: i64,
    evaluations: i64,
    raison: &'static str,
    hstrict_viol: i64,
    hbarrier_viol: i64,
    lp: f64,
}

fn descente_sous_budget(x0: &[i64], poids: &[i64], valeurs: &[i64], capacite: i64, budget: i64, collect: bool) -> Descente {
    let lp = borne_lp(poids, valeurs, capacite);
    let mut x = x0.to_vec();
    let mut phi = -valeur(&x, valeurs);
    let mut trajectoire: Vec<i64> = if collect { vec![phi] } else { vec![] };
    let mut hstrict_viol = 0i64;
    let mut hbarrier_viol = 0i64;
    let mut evaluations = 0i64;
    let mut pas = 0i64;
    let mut blocage = false;
    while evaluations < budget {
        let mut best_delta = 0i64;
        let mut best_i: Option<usize> = None;
        let mut explores = 0i64;
        for i in 0..x.len() {
            if evaluations >= budget { break; }
            explores += 1;
            let mut y = x.clone();
            y[i] = 1 - y[i];
            if !admissible(&y, poids, capacite) { evaluations += 1; continue; }
            evaluations += 1;
            let d = valeur(&y, valeurs) - valeur(&x, valeurs);
            if d > best_delta { best_delta = d; best_i = Some(i); }
        }
        match best_i {
            None => { if explores == x.len() as i64 { blocage = true; } break; }
            Some(bi) => {
                let mut y = x.clone();
                y[bi] = 1 - y[bi];
                let phi_new = -valeur(&y, valeurs);
                if phi_new >= phi { hstrict_viol += 1; }
                if (-phi_new as f64) > lp + 1e-9 { hbarrier_viol += 1; }
                x = y; phi = phi_new; pas += 1;
                if collect { trajectoire.push(phi); }
            }
        }
    }
    let raison = if blocage { "blocage" } else { "budget" };
    Descente { x, valeur: -phi, trajectoire, pas, evaluations, raison, hstrict_viol, hbarrier_viol, lp }
}

// TODO : appeler x0_gen et descendre pour chacun des k departs ;
// repartir le budget entre les appels, puis retourner le meilleur resultat obtenu.
fn descente_restart(
    _x0_gen: &mut dyn FnMut() -> Vec<i64>,
    _poids: &[i64],
    _valeurs: &[i64],
    _capacite: i64,
    _budget_global: i64,
    _k_restarts: i64,
    _descendre: &mut dyn FnMut(&[i64], i64) -> Descente,
) -> Option<Descente> { None }

fn verifier_restarts(
    departs: &mut dyn FnMut() -> Vec<i64>,
    poids: &[i64],
    valeurs: &[i64],
    capacite: i64,
    budget: i64,
    k: i64,
) -> Option<Descente> {
    let fournis = std::cell::RefCell::new(Vec::new());
    let resultats = std::cell::RefCell::new(Vec::new());
    let allocations = std::cell::RefCell::new(Vec::new());

    let mut prochain = || {
        let x = departs();
        assert!(admissible(&x, poids, capacite));
        fournis.borrow_mut().push(x.clone());
        x
    };

    let mut descendre = |x: &[i64], allocation: i64| -> Descente {
        assert!(resultats.borrow().len() < fournis.borrow().len());
        assert!(x == &fournis.borrow()[resultats.borrow().len()]);
        assert!(allocation >= 0);
        allocations.borrow_mut().push(allocation);
        assert!(allocations.borrow().iter().sum::<i64>() <= budget);
        let r = descente_sous_budget(x, poids, valeurs, capacite, allocation, true);
        resultats.borrow_mut().push((r.clone(), r.x.clone(), r.valeur, r.evaluations));
        r
    };

    let r = descente_restart(&mut prochain, poids, valeurs, capacite, budget, k, &mut descendre);
    if r.is_none() { return None; }
    let r = r.unwrap();
    assert!(fournis.borrow().len() == resultats.borrow().len() && resultats.borrow().len() == k as usize);
    assert!(allocations.borrow().iter().sum::<i64>() <= budget);
    assert!(resultats.borrow().iter().map(|&(_, _, _, e)| e).sum::<i64>() <= budget);
    // identite : le resultat retourne est l'un des resultats stockes (par indice)
    let idx = resultats.borrow().iter().position(|(s, _, _, _)| s.x == r.x && s.valeur == r.valeur && s.evaluations == r.evaluations).unwrap();
    assert!(r.x == resultats.borrow()[idx].1 && r.valeur == resultats.borrow()[idx].2);
    assert!(r.valeur == resultats.borrow().iter().map(|&(_, _, v, _)| v).max().unwrap());
    assert!(r.x.len() == poids.len() && r.x.iter().all(|&b| b == 0 || b == 1));
    assert!(admissible(&r.x, poids, capacite) && valeur(&r.x, valeurs) == r.valeur);
    Some(r)
}

fn sequence_departs(sequence: Vec<Vec<i64>>) -> Box<dyn FnMut() -> Vec<i64>> {
    let mut it = sequence.into_iter();
    Box::new(move || it.next().unwrap())
}

{
    let mut seq1 = sequence_departs(vec![vec![0, 0], vec![0, 1]]);
    let r1 = verifier_restarts(&mut seq1, &[2, 1], &[1, 2], 1, 20, 2);
    let mut seq2 = sequence_departs(vec![vec![0], vec![0]]);
    let r2 = verifier_restarts(&mut seq2, &[1], &[7], 0, 10, 2);
    let mut seq3 = sequence_departs(vec![vec![0, 0], vec![0, 1]]);
    let r0 = verifier_restarts(&mut seq3, &[2, 1], &[1, 2], 1, 0, 2);

    if r1.is_none() || r2.is_none() || r0.is_none() {
        println!("Exercice 1 a completer");
    } else {
        let r1 = r1.unwrap(); let r2 = r2.unwrap(); let r0 = r0.unwrap();
        assert!((r1.valeur, r2.valeur, r0.valeur) == (2, 0, 2));
        assert!(r0.evaluations == 0);
        println!("Exercice 1 : appels, budgets et fixtures valides");
        // Meme protocole et budget que la section 3, 60 instances ; aucun gain suppose.
        let mut sans = Vec::new();
        let mut avec = Vec::new();
        let mut categories_sans: Vec<(&str, i64)> = vec![("budget", 0), ("cible", 0), ("hors cible", 0)];
        let mut categories_avec: Vec<(&str, i64)> = vec![("budget", 0), ("cible", 0), ("hors cible", 0)];
        for seed in 0..60 {
            let (p, v, cap) = instance_knapsack(24, seed);
            let p_dep = p.clone();
            let mut rng = Mt19937::new(seed as u32);
            let mut depart = move || {
                let x: Vec<i64> = (0..p_dep.len()).map(|_| rng.randint(0, 1)).collect();
                if admissible(&x, &p_dep, cap) { x } else { vec![0; p_dep.len()] }
            };
            let initial = depart();
            let base = descente_sous_budget(&initial, &p, &v, cap, 2304, false);
            let p_dep2 = p.clone();
            let mut rng2 = Mt19937::new(seed as u32);
            let mut depart2 = move || {
                let x: Vec<i64> = (0..p_dep2.len()).map(|_| rng2.randint(0, 1)).collect();
                if admissible(&x, &p_dep2, cap) { x } else { vec![0; p_dep2.len()] }
            };
            let r = verifier_restarts(&mut depart2, &p, &v, cap, 2304, 4).unwrap();
            let opt = dp_exact(&p, &v, cap);
            for (resultat, categories) in [(&base, &mut categories_sans), (&r, &mut categories_avec)] {
                let cat = if resultat.raison == "budget" { 0 } else if resultat.valeur == opt { 1 } else { 2 };
                categories[cat].1 += 1;
            }
            sans.push(base.valeur as f64 / opt as f64);
            avec.push(r.valeur as f64 / opt as f64);
        }
        let mut sans_s = sans.clone();
        let mut avec_s = avec.clone();
        let med_sans = median(&mut sans_s);
        let med_avec = median(&mut avec_s);
        println!("60 instances, budget 2304 : ratio median sans restart {:.1}%, avec {:.1}%", med_sans * 100.0, med_avec * 100.0);
        println!("arrets sans restart {{'budget': {}, 'cible': {}, 'hors cible': {}}} ; meilleur restart {{'budget': {}, 'cible': {}, 'hors cible': {}}}",
            categories_sans[0].1, categories_sans[1].1, categories_sans[2].1,
            categories_avec[0].1, categories_avec[1].1, categories_avec[2].1);
    }
};

Exercice 1 a completer


## 3. Le contre-claim : `hnostall` ne survit pas au paysage générique

Le théorème Lean conclut « le dernier état est **dans la cible** » — mais seulement sous `hnostall` : *hors de la cible, un mouvement accepté existe toujours*. Cette hypothèse est gratuite en paysage générique. Ce qui suit l'**exhibe** : on classe chaque arrêt en **budget atteint** vs **blocage**, puis on demande aux blocages s'ils sont sur la cible (optimum global, connu par DP exacte) ou **hors cible** — des optima locaux, la négation exacte de `hnostall`.

In [6]:
// LA DISSOCIATION, EXERCICEE : budget-atteint vs blocage, et blocage-hors-cible

struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn instance_knapsack(n: usize, seed: i64) -> (Vec<i64>, Vec<i64>, i64) {
    let mut rng = Mt19937::new(seed as u32);
    let poids: Vec<i64> = (0..n).map(|_| rng.randint(10, 100)).collect();
    let valeurs: Vec<i64> = (0..n).map(|i| poids[i] + rng.randint(-5, 20)).collect();
    let capacite = (0.5 * poids.iter().sum::<i64>() as f64) as i64;
    (poids, valeurs, capacite)
}

fn borne_lp(poids: &[i64], valeurs: &[i64], capacite: i64) -> f64 {
    let n = poids.len();
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&a, &b| (valeurs[b] as f64 / poids[b] as f64).total_cmp(&(valeurs[a] as f64 / poids[a] as f64)));
    let mut reste = capacite;
    let mut total = 0.0f64;
    for &i in &ordre {
        let pris = std::cmp::min(poids[i], reste);
        total += (valeurs[i] * pris) as f64 / poids[i] as f64;
        reste -= pris;
        if reste == 0 { break; }
    }
    total
}

fn dp_exact(poids: &[i64], valeurs: &[i64], capacite: i64) -> i64 {
    let mut best = vec![0i64; (capacite + 1) as usize];
    for (p, v) in poids.iter().zip(valeurs.iter()) {
        for c in (*p as usize..=capacite as usize).rev() {
            if best[c - *p as usize] + v > best[c] {
                best[c] = best[c - *p as usize] + v;
            }
        }
    }
    best[capacite as usize]
}

fn admissible(x: &[i64], poids: &[i64], capacite: i64) -> bool {
    poids.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(p, &_)| *p).sum::<i64>() <= capacite
}

fn valeur(x: &[i64], valeurs: &[i64]) -> i64 {
    valeurs.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(&v, _)| v).sum()
}

#[derive(Clone)]
struct Descente {
    x: Vec<i64>,
    valeur: i64,
    trajectoire: Vec<i64>,
    pas: i64,
    evaluations: i64,
    raison: &'static str,
    hstrict_viol: i64,
    hbarrier_viol: i64,
    lp: f64,
}

fn descente_sous_budget(x0: &[i64], poids: &[i64], valeurs: &[i64], capacite: i64, budget: i64, collect: bool) -> Descente {
    let lp = borne_lp(poids, valeurs, capacite);
    let mut x = x0.to_vec();
    let mut phi = -valeur(&x, valeurs);
    let mut trajectoire: Vec<i64> = if collect { vec![phi] } else { vec![] };
    let mut hstrict_viol = 0i64;
    let mut hbarrier_viol = 0i64;
    let mut evaluations = 0i64;
    let mut pas = 0i64;
    let mut blocage = false;
    while evaluations < budget {
        let mut best_delta = 0i64;
        let mut best_i: Option<usize> = None;
        let mut explores = 0i64;
        for i in 0..x.len() {
            if evaluations >= budget { break; }
            explores += 1;
            let mut y = x.clone();
            y[i] = 1 - y[i];
            if !admissible(&y, poids, capacite) { evaluations += 1; continue; }
            evaluations += 1;
            let d = valeur(&y, valeurs) - valeur(&x, valeurs);
            if d > best_delta { best_delta = d; best_i = Some(i); }
        }
        match best_i {
            None => { if explores == x.len() as i64 { blocage = true; } break; }
            Some(bi) => {
                let mut y = x.clone();
                y[bi] = 1 - y[bi];
                let phi_new = -valeur(&y, valeurs);
                if phi_new >= phi { hstrict_viol += 1; }
                if (-phi_new as f64) > lp + 1e-9 { hbarrier_viol += 1; }
                x = y; phi = phi_new; pas += 1;
                if collect { trajectoire.push(phi); }
            }
        }
    }
    let raison = if blocage { "blocage" } else { "budget" };
    Descente { x, valeur: -phi, trajectoire, pas, evaluations, raison, hstrict_viol, hbarrier_viol, lp }
}

fn median(v: &mut Vec<f64>) -> f64 {
    v.sort_by(|a, b| a.total_cmp(b));
    let n = v.len();
    if n % 2 == 1 { v[n / 2] } else { (v[n / 2 - 1] + v[n / 2]) / 2.0 }
}

let budgets_serre = 4 * 24 * 24; // budget volontairement serre pour rendre 'budget' observable
let n = 24;
let mut compteur_budget = 0i64;
let mut compteur_blocage_cible = 0i64;
let mut compteur_blocage_hc = 0i64;
let mut gaps_hors_cible: Vec<f64> = Vec::new();

for seed in 0..60 {
    let (poids, valeurs, capacite) = instance_knapsack(n, seed);
    let opt = dp_exact(&poids, &valeurs, capacite);
    let mut rng = Mt19937::new(seed as u32);
    let mut x0: Vec<i64> = (0..n).map(|_| rng.randint(0, 1)).collect();
    if !admissible(&x0, &poids, capacite) { x0 = vec![0; n]; }
    let r = descente_sous_budget(&x0, &poids, &valeurs, capacite, budgets_serre as i64, false);
    if r.raison == "budget" {
        compteur_budget += 1;
        gaps_hors_cible.push(1.0 - r.valeur as f64 / opt as f64); // budget epuise : gap quelconque
    } else if r.valeur == opt {
        compteur_blocage_cible += 1;
    } else {
        compteur_blocage_hc += 1;
        gaps_hors_cible.push(1.0 - r.valeur as f64 / opt as f64);
    }
}

let total = compteur_budget + compteur_blocage_cible + compteur_blocage_hc;
println!("runs                        : {}", total);
println!("budget atteint              : {:>3}  ({:.0}%)", compteur_budget, (compteur_budget as f64 / total as f64) * 100.0);
println!("blocage SUR la cible        : {:>3}  ({:.0}%)", compteur_blocage_cible, (compteur_blocage_cible as f64 / total as f64) * 100.0);
println!("blocage HORS cible (optimum local) : {:>3}  ({:.0}%)", compteur_blocage_hc, (compteur_blocage_hc as f64 / total as f64) * 100.0);
let mut gaps_sorted = gaps_hors_cible.clone();
let med = median(&mut gaps_sorted);
let mx = gaps_hors_cible.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
println!("gap final median            : {:.1}%", med * 100.0);
println!("gap final max               : {:.1}%", mx * 100.0);
println!();
print!("=> *[exercee ici]* hnostall est fausse : ");
if compteur_blocage_hc > 0 {
    println!("{} runs se bloquent sur un optimum local :", compteur_blocage_hc);
    println!("   le theoreme 'dernier etat dans la cible' ne se transporte PAS ;");
    println!("   sa conclusion devient : dernier etat = minimum LOCAL, atteint sous budget.");
} else {
    println!("aucun optimum local observe sur cet echantillon (a confronter a un voisinage plus pauvre).");
}

// Regime budget SERRE (~2 balayages) : la cause d'arret 'budget' existe aussi
let mut tight_budget = 0i64;
let mut tight_blocage = 0i64;
for seed in 0..30 {
    let (poids, valeurs, capacite) = instance_knapsack(n, seed);
    let mut rng = Mt19937::new(seed as u32);
    let mut x0: Vec<i64> = (0..n).map(|_| rng.randint(0, 1)).collect();
    if !admissible(&x0, &poids, capacite) { x0 = vec![0; n]; }
    let r = descente_sous_budget(&x0, &poids, &valeurs, capacite, 48, false);
    if r.raison == "budget" { tight_budget += 1; } else { tight_blocage += 1; }
}
println!("a budget serre (48 evals ~ 2 balayages) : budget {}/30, blocage {}/30", tight_budget, tight_blocage);

runs                        : 60


budget atteint              :   0  (0%)


blocage SUR la cible        :   0  (0%)


blocage HORS cible (optimum local) :  60  (100%)


gap final median            : 10.0%


gap final max               : 16.0%


=> *[exercee ici]* hnostall est fausse : 60 runs se bloquent sur un optimum local :


   le theoreme 'dernier etat dans la cible' ne se transporte PAS ;


   sa conclusion devient : dernier etat = minimum LOCAL, atteint sous budget.


a budget serre (48 evals ~ 2 balayages) : budget 25/30, blocage 5/30


### Interprétation — le défaut comme dissociation

Les comptages ci-dessus distinguent les balayages complets sans amélioration des interruptions sur budget. Un blocage hors cible est un témoin contre `hnostall` pour ce voisinage ; un arrêt sur budget ne permet pas de conclure à un optimum local. Les points de départ sont tirés avec un générateur initialisé une seule fois par graine.

Dans cette exécution Rust, 60 blocages sur 60 sont hors cible (gap médian 10,0 %) ; à 48 évaluations, 25 arrêts sur budget et 5 blocages sont observés.

La distinction est celle de deux informations différentes. Après un balayage complet sans améliorant, le voisinage inspecté certifie seulement un optimum local ; la DP exacte permet alors de constater s'il est également global. Après une interruption sur budget, les voisins non examinés peuvent encore contenir un améliorant : ni blocage ni qualité globale ne sont établis. La réserve `hnostall` est ainsi une hypothèse substantielle sur le paysage et le voisinage, et non un effet que l'on peut déduire de la seule terminaison du programme.

### Exercice 2 — barrière : LP fractionnaire vs borne triviale somme des valeurs

**Contexte.** La section 2 vérifie `hbarrier` à chaque mouvement accepté : aucun état ne doit dépasser le majorant LP `lp` calculé par `borne_lp`. La loi de terminaison borne le nombre de pas par la barrière : une barrière plus serrée donne un plafond plus petit.

**Objectif.** Écrire `plafond_theorique(valeur_initiale, borne)` : le majorant du nombre de pas acceptés, à partir de la valeur initiale de `x0` et de la barrière `lp`. Comparer ensuite ce majorant à celui obtenu avec la borne triviale somme des valeurs, sur les mêmes instances.

**Indices.**
- `hbarrier` est une hypothèse : une barrière plus serrée raccourcit le plafond théorique.
- Recalculer le majorant de flips de `descent_flips_le_barrier` (`Descent.lean`) avec `lp`, contre la valeur initiale.
- Comparer au pas réellement observé (champ `pas` renvoyé par `descente_sous_budget`).

**Critère de vérification.** Le majorant est un nombre supérieur ou égal au champ `pas` de chaque run de la section 2 ; un majorant inférieur au pas observé signale une erreur de barrière.

In [7]:
// TODO : retourner le plafond de flips pour une borne entiere
fn plafond_theorique(_valeur_initiale: i64, _borne: i64) -> Option<i64> { None }

let r1 = plafond_theorique(3, 10);
let r2 = plafond_theorique(7, 9);
if r1.is_none() || r2.is_none() {
    println!("Exercice 2 a completer");
} else {
    assert!((r1, r2) == (Some(7), Some(2)));
    println!("Exercice 2 : fixtures valides");
};

Exercice 2 a completer


## 4. Ce que le budget achète : courbe qualité-budget

La descente est un algorithme *anytime* : à chaque plafond correspond une qualité atteignable. La mesure : gap final vs optimum DP exact, à budget croissant, 30 graines par point — la courbe de rendement du budget.

In [8]:
// COURBE QUALITE-BUDGET (anytime) : gap median vs budget, multi-seed

struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn instance_knapsack(n: usize, seed: i64) -> (Vec<i64>, Vec<i64>, i64) {
    let mut rng = Mt19937::new(seed as u32);
    let poids: Vec<i64> = (0..n).map(|_| rng.randint(10, 100)).collect();
    let valeurs: Vec<i64> = (0..n).map(|i| poids[i] + rng.randint(-5, 20)).collect();
    let capacite = (0.5 * poids.iter().sum::<i64>() as f64) as i64;
    (poids, valeurs, capacite)
}

fn borne_lp(poids: &[i64], valeurs: &[i64], capacite: i64) -> f64 {
    let n = poids.len();
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&a, &b| (valeurs[b] as f64 / poids[b] as f64).total_cmp(&(valeurs[a] as f64 / poids[a] as f64)));
    let mut reste = capacite;
    let mut total = 0.0f64;
    for &i in &ordre {
        let pris = std::cmp::min(poids[i], reste);
        total += (valeurs[i] * pris) as f64 / poids[i] as f64;
        reste -= pris;
        if reste == 0 { break; }
    }
    total
}

fn dp_exact(poids: &[i64], valeurs: &[i64], capacite: i64) -> i64 {
    let mut best = vec![0i64; (capacite + 1) as usize];
    for (p, v) in poids.iter().zip(valeurs.iter()) {
        for c in (*p as usize..=capacite as usize).rev() {
            if best[c - *p as usize] + v > best[c] {
                best[c] = best[c - *p as usize] + v;
            }
        }
    }
    best[capacite as usize]
}

fn admissible(x: &[i64], poids: &[i64], capacite: i64) -> bool {
    poids.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(p, &_)| *p).sum::<i64>() <= capacite
}

fn valeur(x: &[i64], valeurs: &[i64]) -> i64 {
    valeurs.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(&v, _)| v).sum()
}

#[derive(Clone)]
struct Descente {
    x: Vec<i64>,
    valeur: i64,
    trajectoire: Vec<i64>,
    pas: i64,
    evaluations: i64,
    raison: &'static str,
    hstrict_viol: i64,
    hbarrier_viol: i64,
    lp: f64,
}

fn descente_sous_budget(x0: &[i64], poids: &[i64], valeurs: &[i64], capacite: i64, budget: i64, collect: bool) -> Descente {
    let lp = borne_lp(poids, valeurs, capacite);
    let mut x = x0.to_vec();
    let mut phi = -valeur(&x, valeurs);
    let mut trajectoire: Vec<i64> = if collect { vec![phi] } else { vec![] };
    let mut hstrict_viol = 0i64;
    let mut hbarrier_viol = 0i64;
    let mut evaluations = 0i64;
    let mut pas = 0i64;
    let mut blocage = false;
    while evaluations < budget {
        let mut best_delta = 0i64;
        let mut best_i: Option<usize> = None;
        let mut explores = 0i64;
        for i in 0..x.len() {
            if evaluations >= budget { break; }
            explores += 1;
            let mut y = x.clone();
            y[i] = 1 - y[i];
            if !admissible(&y, poids, capacite) { evaluations += 1; continue; }
            evaluations += 1;
            let d = valeur(&y, valeurs) - valeur(&x, valeurs);
            if d > best_delta { best_delta = d; best_i = Some(i); }
        }
        match best_i {
            None => { if explores == x.len() as i64 { blocage = true; } break; }
            Some(bi) => {
                let mut y = x.clone();
                y[bi] = 1 - y[bi];
                let phi_new = -valeur(&y, valeurs);
                if phi_new >= phi { hstrict_viol += 1; }
                if (-phi_new as f64) > lp + 1e-9 { hbarrier_viol += 1; }
                x = y; phi = phi_new; pas += 1;
                if collect { trajectoire.push(phi); }
            }
        }
    }
    let raison = if blocage { "blocage" } else { "budget" };
    Descente { x, valeur: -phi, trajectoire, pas, evaluations, raison, hstrict_viol, hbarrier_viol, lp }
}

fn median(v: &mut Vec<f64>) -> f64 {
    v.sort_by(|a, b| a.total_cmp(b));
    let n = v.len();
    if n % 2 == 1 { v[n / 2] } else { (v[n / 2 - 1] + v[n / 2]) / 2.0 }
}

let n = 24;
let mut echantillons: Vec<(i64, Vec<f64>)> = Vec::new();
for B in [20i64, 50, 100, 200, 400, 800] {
    let mut gaps: Vec<f64> = Vec::new();
    for seed in 0..30 {
        let (poids, valeurs, capacite) = instance_knapsack(n, 1000 + seed);
        let opt = dp_exact(&poids, &valeurs, capacite);
        let mut rng = Mt19937::new((2000 + seed) as u32);
        let mut x0: Vec<i64> = (0..n).map(|_| rng.randint(0, 1)).collect();
        if !admissible(&x0, &poids, capacite) { x0 = vec![0; n]; }
        let r = descente_sous_budget(&x0, &poids, &valeurs, capacite, B, false);
        gaps.push(1.0 - r.valeur as f64 / opt as f64);
    }
    let mut q = gaps.clone();
    q.sort_by(|a, b| a.total_cmp(b));
    let med = median(&mut gaps);
    let med_str = format!("{:>6}", format!("{:.1}%", med * 100.0));
    let q7_str = format!("{:.1}%", q[7] * 100.0);
    let q22_str = format!("{:.1}%", q[22] * 100.0);
    let pire_str = format!("{:.1}%", q[29] * 100.0);
    println!("budget {:>4} evaluations | gap median {} | IQR [{}, {}] | pire {}", B, med_str, q7_str, q22_str, pire_str);
    echantillons.push((B, gaps));
}

let g_med: Vec<f64> = echantillons.iter().map(|(_, g)| {
    let mut gg = g.clone();
    median(&mut gg)
}).collect();
let ameliorations: Vec<f64> = (0..g_med.len() - 1).map(|i| g_med[i] - g_med[i + 1]).collect();
println!();
println!("rendement marginal (gain de gap par doubling de budget) :");
for i in 0..ameliorations.len() {
    println!("  {:>4} -> {:>4} : {:+.1}%", echantillons[i].0, echantillons[i + 1].0, ameliorations[i] * 100.0);
};

budget   20 evaluations | gap median  86.3% | IQR [18.4%, 87.1%] | pire 89.0%


budget   50 evaluations | gap median  63.8% | IQR [12.2%, 66.4%] | pire 72.6%


budget  100 evaluations | gap median  39.3% | IQR [12.2%, 42.8%] | pire 48.0%


budget  200 evaluations | gap median  10.8% | IQR [7.9%, 12.7%] | pire 17.0%


budget  400 evaluations | gap median   8.9% | IQR [7.0%, 12.2%] | pire 17.0%


budget  800 evaluations | gap median   8.9% | IQR [7.0%, 12.2%] | pire 17.0%


... (2 outputs left out)

    50 ->  100 : +24.5%


   100 ->  200 : +28.5%


   200 ->  400 : +1.9%


   400 ->  800 : +0.0%


### Interprétation

Les statistiques ci-dessus décrivent les 30 instances et ces budgets ; elles ne constituent pas une garantie générale de rendement décroissant. Un plateau peut provenir de blocages locaux ; il faut examiner les raisons d'arrêt pour l'établir. Changer de voisinage ou redémarrer peut modifier la qualité à budget égal.

Le gap médian mesuré passe de 86,3 % à 20 évaluations à 8,9 % à 400 puis 800 évaluations.

### Exercice 3 — voisinage un ou deux flips

Étendre la descente aux flips simples **et** aux paires, sous le même budget. L'union contient le voisinage à un flip ; un optimum local à un flip peut rester local pour le voisinage élargi, ou cesser de l'être. Pour chaque candidat, appeler `evaluer(depuis, voisin)`, qui compte effectivement l'évaluation et retourne la valeur ou `None` si le voisin est inadmissible. Retourner un dictionnaire avec `x`, `valeur`, `chemin` (liste des états acceptés, départ inclus), `evaluations` et `raison` (`budget` ou `blocage`).

Le contrôle vérifie états et valeurs recalculées, trace des mouvements évalués, budget nul, capacité nulle et certificats de blocage. Une fois le stub complété, il retrace la courbe sur les six budgets et les 30 graines de la section 4.

In [9]:
// Stub C.1 — Exercice 3 : voisinage un ou deux flips

struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn instance_knapsack(n: usize, seed: i64) -> (Vec<i64>, Vec<i64>, i64) {
    let mut rng = Mt19937::new(seed as u32);
    let poids: Vec<i64> = (0..n).map(|_| rng.randint(10, 100)).collect();
    let valeurs: Vec<i64> = (0..n).map(|i| poids[i] + rng.randint(-5, 20)).collect();
    let capacite = (0.5 * poids.iter().sum::<i64>() as f64) as i64;
    (poids, valeurs, capacite)
}

fn dp_exact(poids: &[i64], valeurs: &[i64], capacite: i64) -> i64 {
    let mut best = vec![0i64; (capacite + 1) as usize];
    for (p, v) in poids.iter().zip(valeurs.iter()) {
        for c in (*p as usize..=capacite as usize).rev() {
            if best[c - *p as usize] + v > best[c] {
                best[c] = best[c - *p as usize] + v;
            }
        }
    }
    best[capacite as usize]
}

fn admissible(x: &[i64], poids: &[i64], capacite: i64) -> bool {
    poids.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(p, &_)| *p).sum::<i64>() <= capacite
}

fn valeur(x: &[i64], valeurs: &[i64]) -> i64 {
    valeurs.iter().zip(x.iter()).filter(|&(_, &xi)| xi != 0).map(|(&v, _)| v).sum()
}

fn median(v: &mut Vec<f64>) -> f64 {
    v.sort_by(|a, b| a.total_cmp(b));
    let n = v.len();
    if n % 2 == 1 { v[n / 2] } else { (v[n / 2 - 1] + v[n / 2]) / 2.0 }
}

struct Resultat2Flip {
    x: Vec<i64>,
    valeur: i64,
    chemin: Vec<Vec<i64>>,
    evaluations: i64,
    raison: &'static str,
}

// TODO : evaluer(depuis, voisin) compte une evaluation et renvoie sa valeur
// ou None si inadmissible. Retourner x, valeur, chemin, evaluations, raison.
fn descente_2flip(
    _x0: &[i64],
    _poids: &[i64],
    _valeurs: &[i64],
    _capacite: i64,
    _budget: i64,
    _evaluer: &mut dyn FnMut(&[i64], &[i64]) -> Option<i64>,
) -> Option<Resultat2Flip> { None }

fn verifier_2flip(x0: &[i64], poids: &[i64], valeurs: &[i64], capacite: i64, budget: i64) -> Option<Resultat2Flip> {
    let evenements = std::cell::RefCell::new(Vec::new());

    let mut evaluer = |depuis: &[i64], voisin: &[i64]| -> Option<i64> {
        assert!(evenements.borrow().len() < budget as usize);
        let u = depuis.to_vec();
        let v = voisin.to_vec();
        assert!(u.len() == v.len() && u.len() == poids.len());
        assert!(u.iter().chain(v.iter()).all(|&b| b == 0 || b == 1) && admissible(&u, poids, capacite));
        let dist = u.iter().zip(v.iter()).filter(|&(a, b)| a != b).count();
        assert!(1 <= dist && dist <= 2);
        let val = if admissible(&v, poids, capacite) { Some(valeur(&v, valeurs)) } else { None };
        evenements.borrow_mut().push((u, v));
        val
    };

    let r = descente_2flip(x0, poids, valeurs, capacite, budget, &mut evaluer);
    if r.is_none() { return None; }
    let r = r.unwrap();
    let chemin = &r.chemin;
    assert!(!chemin.is_empty() && chemin[0] == x0.to_vec() && *chemin.last().unwrap() == r.x);
    assert!(r.evaluations as usize == evenements.borrow().len() && evenements.borrow().len() <= budget as usize);
    assert!(r.raison == "budget" || r.raison == "blocage");
    for x in chemin {
        assert!(x.len() == poids.len() && x.iter().all(|&b| b == 0 || b == 1) && admissible(x, poids, capacite));
    }
    assert!(valeur(&r.x, valeurs) == r.valeur);
    let mut debut = 0usize;
    for pair in chemin.windows(2) {
        let (u, v) = (&pair[0], &pair[1]);
        let dist = u.iter().zip(v.iter()).filter(|&(a, b)| a != b).count();
        assert!(1 <= dist && dist <= 2);
        assert!(valeur(v, valeurs) > valeur(u, valeurs));
        let ev = evenements.borrow();
        let indice = (debut..ev.len()).find(|&i| ev[i].0 == *u && ev[i].1 == *v);
        assert!(indice.is_some());
        debut = indice.unwrap() + 1;
    }
    if r.raison == "blocage" {
        // Un certificat requiert tous les voisins depuis l'etat final.
        let x = &r.x;
        let ev = evenements.borrow();
        let vus: std::collections::HashSet<Vec<i64>> = ev[debut..].iter().filter(|(u, _)| u == x).map(|(_, v)| v.clone()).collect();
        let mut attendus: std::collections::HashSet<Vec<i64>> = std::collections::HashSet::new();
        for i in 0..x.len() {
            let mut y = x.clone();
            y[i] ^= 1;
            attendus.insert(y);
        }
        for i in 0..x.len() {
            for j in (i + 1)..x.len() {
                let mut y = x.clone();
                y[i] ^= 1;
                y[j] ^= 1;
                attendus.insert(y);
            }
        }
        for a in &attendus {
            assert!(vus.contains(a));
        }
        for y in &attendus {
            assert!(!admissible(y, poids, capacite) || valeur(y, valeurs) <= r.valeur);
        }
    } else {
        assert!(evenements.borrow().len() == budget as usize);
    }
    Some(r)
}

let a = verifier_2flip(&[0], &[1], &[5], 1, 5);
let b = verifier_2flip(&[1, 0], &[1, 1], &[1, 4], 1, 10);
let c = verifier_2flip(&[0, 0], &[1, 2], &[3, 4], 0, 10);
let z = verifier_2flip(&[0, 0], &[2, 1], &[1, 2], 1, 0);
if a.is_none() || b.is_none() || c.is_none() || z.is_none() {
    println!("Exercice 3 a completer");
} else {
    let a = a.unwrap(); let b = b.unwrap(); let c = c.unwrap(); let z = z.unwrap();
    assert!((a.valeur, b.valeur, c.valeur, z.valeur) == (5, 4, 0, 0));
    assert!(z.evaluations == 0 && z.raison == "budget");
    println!("Exercice 3 : traces observees, budgets et fixtures valides");
    for budget in [20i64, 50, 100, 200, 400, 800] {
        let mut gaps: Vec<f64> = Vec::new();
        for seed in 0..30 {
            let (p, v, cap) = instance_knapsack(24, 1000 + seed);
            let mut rng = Mt19937::new((2000 + seed) as u32);
            let mut x: Vec<i64> = (0..p.len()).map(|_| rng.randint(0, 1)).collect();
            if !admissible(&x, &p, cap) { x = vec![0; p.len()]; }
            let r = verifier_2flip(&x, &p, &v, cap, budget).unwrap();
            gaps.push(1.0 - r.valeur as f64 / dp_exact(&p, &v, cap) as f64);
        }
        let mut gs = gaps.clone();
        let med = median(&mut gs);
        println!("1 ou 2 flips : budget {}, gap median {:.1}%", budget, med * 100.0);
    }
};

Exercice 3 a completer


## 5. Expérience ICT candidate, limites, exercices

**Expérience ICT candidate** : le protocole qualité-budget de §4, appliqué à une descente **multi-regards** (op 12) — budget partagé entre deux potentiels différents, courbe comparée au budget intégrairement alloué au regard le plus performant en isolation. Tester si le partage améliore ou non le meilleur regard isolé est une hypothèse expérimentale ; les observations de §3 ne tranchent pas cette comparaison.

**Limites** (bornes explicites) : (1) substrat sac à dos 0/1 seul — voisinage bit-flip unimodal par objet ; (2) hill-climbing déterministe — pas de recuit, pas de restarts (c'est l'exercice 1, pas le corps) ; (3) la comparaison MEALPy reste le territoire de Search-11 — ce notebook énonce la loi, il ne refait pas le tour d'horizon.

**Exercices** — les stubs s'exécutent sans erreur (politique du dépôt : compléter, pas débloquer).

## Conclusion

| Exigence de distillation | Où elle vit |
|---|---|
| Source primaire firsthand | `mimo_lean/Descent.lean` — `descent_flips_le_barrier` l.110, `descent_target_before_ceiling` l.144, thèse op 11 l.148 |
| Objet formel | `descente_sous_budget` : potentiel strictement décroissant, barrière LP, plafond d'évaluations |
| Claim exact, exercé | §2 : 60 exécutions, 0 violation `hstrict`, 0 violation `hbarrier`, 0 dépassement de plafond — *[exercée ici]* |
| Contre-claim, exhibé | §3 : `hnostall` fausse — blocages hors cible et arrêts sur budget mesurés dans les sorties |
| Expérience ICT candidate | §5 : qualité-budget d'une descente multi-regards (op 12) |

**Attestation** : 2ᵉ attestation de l'opération 11 sur un substrat indépendant (Rust std, stdout réels) — la loi est exercée sur un second langage, confirmant que le claim ne dépend pas de l'implémentation Python.

## À retenir

Un balayage complet sans amélioration établit un optimum local, et non global. Un budget d'évaluations épuisé ne certifie pas la terminalité. Le coût naturel `floor(lp) - valeur` borne les flips acceptés.

## Vérifiez votre compréhension

<details><summary>Pourquoi un balayage partiel ne suffit-il pas ?</summary>Un voisin améliorant peut ne pas avoir été examiné.</details>

<details><summary>Quel compteur borne le résultat Lean ?</summary>Les flips acceptés, pas les évaluations de voisins.</details>

## Pour aller plus loin

Lire `mimo_lean/Descent.lean` pour les hypothèses formelles et Search-11 pour les métaheuristiques. Reprendre les trois exercices : redémarrages, barrière et voisinage élargi.

***

## Références et provenance

> Carnet issu de l'EPIC #12204.